<a href="https://colab.research.google.com/github/1kaiser/Media-Segment-Depth-MLP/blob/main/2D_Positional_Encoding_Vision_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## [RoPE](https://arxiv.org/pdf/2410.06205v1) vs [STRING Position Encoding](https://arxiv.org/pdf/2502.02562) Visualization for Images

In [ ]:
!curl "https://i.imgur.com/U66BzSa.jpeg" --output dogs.jpg

In [ ]:
"""
RoPE vs STRING Position Encoding Visualization for Images
Run this in your notebook environment after downloading the image in a separate cell.
"""

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
from torchvision import transforms
import torch.nn.functional as F
import math
import os

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

#########################
# 1. RoPE Implementation
#########################

class RoPE2D:
    """2D Rotary Position Embedding for images"""
    def __init__(self, dim, max_freq=10000):
        self.dim = dim
        self.max_freq = max_freq

        # Create frequency bands
        freqs = torch.exp(-torch.arange(0, dim, 4) * np.log(max_freq) / dim)
        self.register_buffer = freqs

    def __call__(self, patches, patch_positions):
        """
        Apply RoPE to patches
        patches: [batch, num_patches, dim]
        patch_positions: [num_patches, 2] containing (row, col) positions
        """
        batch_size, num_patches, dim = patches.shape
        device = patches.device

        # Generate frequencies
        freqs = torch.exp(-torch.arange(0, dim, 4, device=device, dtype=torch.float32) * np.log(self.max_freq) / dim)

        # Get x and y positions
        pos_x = patch_positions[:, 0].float()  # [num_patches]
        pos_y = patch_positions[:, 1].float()

        # Compute rotation angles
        angles_x = pos_x.unsqueeze(-1) * freqs  # [num_patches, dim//4]
        angles_y = pos_y.unsqueeze(-1) * freqs

        # Create rotation matrices for 2D
        cos_x = torch.cos(angles_x)
        sin_x = torch.sin(angles_x)
        cos_y = torch.cos(angles_y)
        sin_y = torch.sin(angles_y)

        # Apply rotations (simplified for visualization)
        rotated_patches = patches.clone()

        # Process each group of 4 dimensions
        for i in range(0, dim, 4):
            if i + 3 < dim:
                # Apply 2D rotation to groups of 4
                rotated_patches[..., i:i+2] = self._rotate_2d(
                    patches[..., i:i+2], cos_x[:, i//4], sin_x[:, i//4]
                )
                rotated_patches[..., i+2:i+4] = self._rotate_2d(
                    patches[..., i+2:i+4], cos_y[:, i//4], sin_y[:, i//4]
                )

        return rotated_patches

    def _rotate_2d(self, x, cos, sin):
        """Apply 2D rotation"""
        x1, x2 = x[..., 0], x[..., 1]
        # Reshape cos and sin to broadcast correctly with the patches tensor
        # cos and sin have shape [num_patches], we need [1, num_patches] for batch dimension
        cos = cos.unsqueeze(0)  # [num_patches] -> [1, num_patches]
        sin = sin.unsqueeze(0)  # [num_patches] -> [1, num_patches]
        return torch.stack([
            x1 * cos - x2 * sin,
            x1 * sin + x2 * cos
        ], dim=-1)

#########################
# 2. STRING Implementation (Simplified)
#########################

class SimpleSTRING:
    """Simplified STRING implementation for visualization"""
    def __init__(self, dim, num_coords=2):
        self.dim = dim
        self.num_coords = num_coords

        # Initialize learnable generators (simplified)
        self.generators = []
        for _ in range(num_coords):
            # Create a skew-symmetric matrix
            A = torch.randn(dim, dim) * 0.01
            skew = A - A.T
            self.generators.append(skew)

        # Learnable orthogonal transformation (Cayley parameterization)
        S = torch.randn(dim, dim) * 0.01
        self.skew_param = S - S.T

    def __call__(self, patches, patch_positions):
        """
        Apply STRING to patches
        patches: [batch, num_patches, dim]
        patch_positions: [num_patches, 2] containing (row, col) positions
        """
        batch_size, num_patches, dim = patches.shape
        device = patches.device

        # Move generators to device
        generators = [g.to(device) for g in self.generators]
        skew_param = self.skew_param.to(device)

        # Compute orthogonal matrix using Cayley transform
        I = torch.eye(dim, device=device)
        P = torch.matmul(I - skew_param, torch.inverse(I + skew_param))

        # Transform patches to learned basis
        transformed_patches = torch.matmul(patches, P.T)

        # Apply position-dependent rotations
        rotated_patches = torch.zeros_like(transformed_patches)

        for idx in range(num_patches):
            # Get position
            pos = patch_positions[idx].float()

            # Compute combined generator
            combined_gen = torch.zeros(dim, dim, device=device)
            for coord_idx, coord_val in enumerate(pos):
                if coord_idx < len(generators):
                    combined_gen += coord_val * generators[coord_idx]

            # Matrix exponential (using truncated series for efficiency)
            rotation = self._matrix_exp_truncated(combined_gen, order=3)

            # Apply rotation to this patch across the batch
            rotated_patches[:, idx] = torch.matmul(transformed_patches[:, idx], rotation.T)

        # Transform back to original basis
        final_patches = torch.matmul(rotated_patches, P)

        return final_patches

    def _matrix_exp_truncated(self, A, order=3):
        """Truncated matrix exponential for efficiency"""
        I = torch.eye(A.shape[0], device=A.device)
        result = I
        A_power = I
        for n in range(1, order + 1):
            A_power = torch.matmul(A_power, A)
            result += A_power / math.factorial(n)

        return result

#########################
# 3. Image Processing and Visualization
#########################

def load_and_preprocess_image(image_path):
    """Load image from a local file, crop it, and preprocess"""
    try:
        # Transformation pipeline for the model (includes crop)
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        # Transformation pipeline for display (just crop, no normalization)
        display_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
        ])

        img = Image.open(image_path).convert('RGB')

        img_tensor = transform(img).unsqueeze(0)
        display_img = display_transform(img)

        return img_tensor, display_img

    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}. Please run the download command first.")
        return None, None
    except Exception as e:
        print(f"An error occurred while loading the image: {e}")
        return None, None


def extract_patches(image_tensor, patch_size=16):
    """Extract patches from image"""
    batch_size, channels, height, width = image_tensor.shape

    # Unfold image into patches
    patches = image_tensor.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    patches = patches.contiguous().view(batch_size, channels, -1, patch_size, patch_size)
    patches = patches.permute(0, 2, 1, 3, 4)  # [batch, num_patches, channels, patch_size, patch_size]

    # Flatten patches
    num_patches = patches.shape[1]
    patches_flat = patches.reshape(batch_size, num_patches, -1)  # [batch, num_patches, channels*patch_size*patch_size]

    # Create position indices
    grid_size = height // patch_size
    positions = []
    for i in range(grid_size):
        for j in range(grid_size):
            positions.append([i, j])
    positions = torch.tensor(positions)

    return patches_flat, positions, patches

def visualize_encodings(original_patches_structured, rope_patches_flat, string_patches_flat, positions, num_show=9):
    """Visualize the effect of position encodings"""
    fig, axes = plt.subplots(3, num_show, figsize=(15, 6))

    indices = np.linspace(0, len(positions)-1, num_show, dtype=int)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    original_patches_flat = original_patches_structured.reshape(1, -1, 3*16*16)

    for i, idx in enumerate(indices):
        # Original patch
        patch_structured = original_patches_structured[0, idx]
        patch_display = patch_structured.unsqueeze(0) * std + mean
        patch_display = torch.clamp(patch_display, 0, 1).squeeze(0)

        axes[0, i].imshow(patch_display.permute(1, 2, 0).cpu().numpy())
        axes[0, i].set_title(f'Pos ({positions[idx, 0].item()}, {positions[idx, 1].item()})')
        axes[0, i].axis('off')

        # RoPE encoded patch (show difference)
        rope_diff = rope_patches_flat[0, idx] - original_patches_flat[0, idx]
        rope_heatmap = rope_diff.reshape(3, 16, 16).mean(0).cpu().detach().numpy()
        im1 = axes[1, i].imshow(rope_heatmap, cmap='RdBu', vmin=-0.5, vmax=0.5)
        axes[1, i].set_title('RoPE Effect')
        axes[1, i].axis('off')

        # STRING encoded patch (show difference)
        string_diff = string_patches_flat[0, idx] - original_patches_flat[0, idx]
        string_heatmap = string_diff.reshape(3, 16, 16).mean(0).cpu().detach().numpy()
        im2 = axes[2, i].imshow(string_heatmap, cmap='RdBu', vmin=-0.5, vmax=0.5)
        axes[2, i].set_title('STRING Effect')
        axes[2, i].axis('off')

    axes[0, 0].set_ylabel('Original\nPatches', fontsize=12)
    axes[1, 0].set_ylabel('RoPE\nDifference', fontsize=12)
    axes[2, 0].set_ylabel('STRING\nDifference', fontsize=12)

    fig.colorbar(im1, ax=axes[1, :], orientation='horizontal', fraction=0.1, pad=0.2)
    fig.colorbar(im2, ax=axes[2, :], orientation='horizontal', fraction=0.1, pad=0.2)

    plt.suptitle('Comparison of Positional Encoding Effects on Image Patches', fontsize=16)
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    return fig

def plot_position_encoding_patterns(rope_encoder, string_encoder, dim=768):
    """Visualize the encoding patterns for different positions"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    grid_size = 14
    num_positions = grid_size * grid_size

    # A single dummy patch that will be copied for all positions
    dummy_patch_base = torch.randn(1, 1, dim)
    # Create a single batch item with 196 patches
    dummy_patches = dummy_patch_base.repeat(1, num_positions, 1)

    # Create the grid of positions
    pos_grid = torch.stack(torch.meshgrid(torch.arange(grid_size), torch.arange(grid_size), indexing='ij')).view(2, -1).T

    with torch.no_grad():
        # Pass the single batch item with 196 patches
        rope_out = rope_encoder(dummy_patches, pos_grid)
        string_out = string_encoder(dummy_patches, pos_grid)

    # Calculate the norm difference. The original patch will be broadcast correctly.
    encoded_rope = torch.norm(rope_out - dummy_patch_base, dim=2).squeeze().cpu().numpy()
    encoded_string = torch.norm(string_out - dummy_patch_base, dim=2).squeeze().cpu().numpy()

    im1 = ax1.imshow(encoded_rope.reshape(grid_size, grid_size), cmap='viridis')
    ax1.set_title('RoPE Encoding Strength by Position')
    ax1.set_xlabel('Patch Column')
    ax1.set_ylabel('Patch Row')
    fig.colorbar(im1, ax=ax1)

    im2 = ax2.imshow(encoded_string.reshape(grid_size, grid_size), cmap='viridis')
    ax2.set_title('STRING Encoding Strength by Position')
    ax2.set_xlabel('Patch Column')
    ax2.set_ylabel('Patch Row')
    fig.colorbar(im2, ax=ax2)

    plt.tight_layout()
    return fig

#########################
# 4. Main Execution
#########################

def main():
    print("🚀 RoPE vs STRING Position Encoding Visualization")
    print("=" * 50)

    IMAGE_FILENAME = "dogs.jpg"

    print(f"📸 Loading, cropping, and preprocessing '{IMAGE_FILENAME}'...")
    img_tensor, display_img = load_and_preprocess_image(IMAGE_FILENAME)

    if img_tensor is None:
        return

    plt.figure(figsize=(5, 5))
    plt.imshow(display_img)
    plt.title("Input Image (Cropped to 224x224)")
    plt.axis('off')
    plt.show()

    print("🔲 Extracting 16x16 patches...")
    patches_flat, positions, patches_structured = extract_patches(img_tensor)
    print(f"   - Number of patches: {patches_flat.shape[1]}")
    print(f"   - Patch dimension: {patches_flat.shape[2]}")

    dim = patches_flat.shape[2]
    rope_encoder = RoPE2D(dim)
    string_encoder = SimpleSTRING(dim)

    print("\n🔄 Applying position encodings...")
    with torch.no_grad():
        rope_patches = rope_encoder(patches_flat, positions)
        string_patches = string_encoder(patches_flat, positions)

    rope_change = torch.norm(rope_patches - patches_flat) / torch.norm(patches_flat)
    string_change = torch.norm(string_patches - patches_flat) / torch.norm(patches_flat)

    print(f"\n📊 Encoding Statistics:")
    print(f"   - RoPE relative change: {rope_change:.4f}")
    print(f"   - STRING relative change: {string_change:.4f}")

    print("\n🎨 Visualizing encoding effects...")
    viz_fig = visualize_encodings(patches_structured, rope_patches, string_patches, positions)
    plt.show()

    print("\n📈 Visualizing position encoding patterns across image...")
    pattern_fig = plot_position_encoding_patterns(rope_encoder, string_encoder, dim)
    plt.show()

    print("\n🤖 What gets fed to a Vision Transformer:")
    print(f"   - Original patches shape: {patches_flat.shape}")
    print(f"   - After RoPE encoding: {rope_patches.shape}")
    print(f"   - After STRING encoding: {string_patches.shape}")

    return patches_flat, rope_patches, string_patches, positions

# Run the visualization
if __name__ == "__main__":
    # In your notebook, run this in a cell:
    # !curl "https://i.imgur.com/U66BzSa.jpeg" --output dogs.jpg

    results = main()

    if results:
        original, rope_encoded, string_encoded, positions = results
        print("\n🔍 Detailed Analysis of Center Patch:")
        center_idx = len(positions) // 2
        center_pos = positions[center_idx]
        print(f"   Position: ({center_pos[0].item()}, {center_pos[1].item()})")

        original_patch = original[0, center_idx]
        rope_patch = rope_encoded[0, center_idx]
        string_patch = string_encoded[0, center_idx]

        cos_sim_rope = F.cosine_similarity(original_patch.unsqueeze(0), rope_patch.unsqueeze(0))
        cos_sim_string = F.cosine_similarity(original_patch.unsqueeze(0), string_patch.unsqueeze(0))

        print(f"   Cosine similarity to original (RoPE): {cos_sim_rope.item():.4f}")
        print(f"   Cosine similarity to original (STRING): {cos_sim_string.item():.4f}")

    print("\n✅ Visualization complete!")

In [ ]:
"""
Simplified Cayley-STRING vs RoPE Demo
Perfect for quick integration and testing
"""

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.gridspec as gridspec

class RoPE2D(nn.Module):
    """2D Rotary Position Embedding"""
    def __init__(self, dim, max_freq=10000):
        super().__init__()
        freqs = torch.exp(-torch.arange(0, dim, 4) * np.log(max_freq) / dim)
        self.register_buffer('freqs', freqs)

    def forward(self, x, pos):
        """x: [B, N, D], pos: [N, 2]"""
        angles_x = pos[:, 0:1] * self.freqs  # [N, D//4]
        angles_y = pos[:, 1:2] * self.freqs

        cos_x, sin_x = torch.cos(angles_x), torch.sin(angles_x)
        cos_y, sin_y = torch.cos(angles_y), torch.sin(angles_y)

        x_rot = x.clone()
        for i in range(0, x.shape[-1], 4):
            if i + 3 < x.shape[-1]:
                # X rotation
                x1, x2 = x[..., i], x[..., i+1]
                x_rot[..., i] = x1 * cos_x[:, i//4] - x2 * sin_x[:, i//4]
                x_rot[..., i+1] = x1 * sin_x[:, i//4] + x2 * cos_x[:, i//4]

                # Y rotation
                y1, y2 = x[..., i+2], x[..., i+3]
                x_rot[..., i+2] = y1 * cos_y[:, i//4] - y2 * sin_y[:, i//4]
                x_rot[..., i+3] = y1 * sin_y[:, i//4] + y2 * cos_y[:, i//4]

        return x_rot

class CayleySTRING(nn.Module):
    """Cayley-STRING: RoPE + Learnable Orthogonal Transform"""
    def __init__(self, dim, max_freq=10000):
        super().__init__()
        self.rope = RoPE2D(dim, max_freq)

        # Learnable antisymmetric matrix for Cayley transform
        self.A = nn.Parameter(torch.randn(dim, dim) * 0.01)

    def get_orthogonal_matrix(self):
        """Cayley transform: P = (I - S)(I + S)^-1 where S = A - A^T"""
        S = self.A - self.A.T  # Antisymmetric
        I = torch.eye(S.shape[0], device=S.device)
        return torch.matmul(I - S, torch.inverse(I + S))

    def forward(self, x, pos):
        """x: [B, N, D], pos: [N, 2]"""
        # Apply RoPE first
        x_rope = self.rope(x, pos)

        # Apply learnable orthogonal transform
        P = self.get_orthogonal_matrix()
        return torch.matmul(x_rope, P.T)

def load_and_preprocess_image(image_path):
    """Load image from a local file, crop it, and preprocess"""
    try:
        # Transformation pipeline for the model (includes crop)
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        # Transformation pipeline for display (just crop, no normalization)
        display_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
        ])

        img = Image.open(image_path).convert('RGB')

        img_tensor = transform(img).unsqueeze(0)
        display_img = display_transform(img)

        return img_tensor, display_img

    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}. Please run the download command first.")
        return None, None
    except Exception as e:
        print(f"An error occurred while loading the image: {e}")
        return None, None

def extract_patches(image_tensor, patch_size=16):
    """Extract patches from image"""
    batch_size, channels, height, width = image_tensor.shape

    # Unfold image into patches
    patches = image_tensor.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    patches = patches.contiguous().view(batch_size, channels, -1, patch_size, patch_size)
    patches = patches.permute(0, 2, 1, 3, 4)  # [batch, num_patches, channels, patch_size, patch_size]

    # Flatten patches
    num_patches = patches.shape[1]
    patches_flat = patches.reshape(batch_size, num_patches, -1)  # [batch, num_patches, channels*patch_size*patch_size]

    # Create position indices
    grid_size = height // patch_size
    positions = []
    for i in range(grid_size):
        for j in range(grid_size):
            positions.append([i, j])
    positions = torch.tensor(positions)

    return patches_flat, positions, patches

def visualize_input_and_patch_grids(display_img, original_patches_structured, rope_patches_flat, string_patches_flat, positions):
    """
    Visualize: INPUT IMAGE | INPUT PATCH GRID | ROPE PATCH GRID | STRING PATCH GRID
    Args:
        display_img (PIL.Image.Image): The original image (cropped to 224x224) for display.
        original_patches_structured (torch.Tensor): The original patches [B, N, C, P, P].
        rope_patches_flat (torch.Tensor): RoPE-encoded patches [B, N, D].
        string_patches_flat (torch.Tensor): STRING-encoded patches [B, N, D].
        positions (torch.Tensor): The 2D grid positions [N, 2].
    """
    batch_size, num_patches, channels, patch_size, _ = original_patches_structured.shape
    grid_size = int(np.sqrt(num_patches))  # Should be 14 for 196 patches

    # Reshape flattened encoded patches back to structured form [B, N, C, P, P]
    rope_patches_structured = rope_patches_flat.reshape(batch_size, num_patches, channels, patch_size, patch_size)
    string_patches_structured = string_patches_flat.reshape(batch_size, num_patches, channels, patch_size, patch_size)

    # Denormalize patches for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1).to(original_patches_structured.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1).to(original_patches_structured.device)

    original_patches_display = torch.clamp(original_patches_structured * std + mean, 0, 1)
    rope_patches_display = torch.clamp(rope_patches_structured * std + mean, 0, 1)
    string_patches_display = torch.clamp(string_patches_structured * std + mean, 0, 1)

    # Create figure with 4 subplots: 1 for image + 3 for patch grids
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # 1. INPUT IMAGE (224x224)
    axes[0].imshow(display_img)
    axes[0].set_title('INPUT IMAGE\n224 x 224', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Helper function to create patch grid
    def create_patch_grid(patches_display, ax, title):
        # Create a canvas to assemble all patches
        patch_canvas = torch.zeros(3, grid_size * patch_size, grid_size * patch_size)

        for i in range(grid_size):
            for j in range(grid_size):
                idx = i * grid_size + j
                patch = patches_display[0, idx]  # [3, 16, 16]

                # Place patch in the canvas
                y_start, y_end = i * patch_size, (i + 1) * patch_size
                x_start, x_end = j * patch_size, (j + 1) * patch_size
                patch_canvas[:, y_start:y_end, x_start:x_end] = patch

        # Display the assembled grid
        ax.imshow(patch_canvas.permute(1, 2, 0).cpu().numpy())
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')

        # Add grid lines to show patch boundaries
        for i in range(1, grid_size):
            ax.axhline(y=i * patch_size - 0.5, color='white', linewidth=0.5, alpha=0.7)
            ax.axvline(x=i * patch_size - 0.5, color='white', linewidth=0.5, alpha=0.7)

    # 2. INPUT PATCH GRID (14x14 tiles of 16x16 patches)
    create_patch_grid(original_patches_display, axes[1],
                     'INPUT PATCH GRID\n14 x 14 tiles\n(16x16 patches)')

    # 3. ROPE PATCH GRID (14x14 tiles of 16x16 patches)
    create_patch_grid(rope_patches_display, axes[2],
                     'ROPE PATCH GRID\n14 x 14 tiles\n(16x16 patches)')

    # 4. STRING PATCH GRID (14x14 tiles of 16x16 patches)
    create_patch_grid(string_patches_display, axes[3],
                     'STRING PATCH GRID\n14 x 14 tiles\n(16x16 patches)')

    plt.tight_layout()
    plt.suptitle('Cayley-STRING vs RoPE: Visual Comparison of Positional Encoding Effects',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.show()

    return fig

def visualize_encoding_differences(display_img, original_patches_structured, rope_patches_flat, string_patches_flat, positions):
    """
    Show the differences/effects of each encoding method as heatmaps
    """
    batch_size, num_patches, channels, patch_size, _ = original_patches_structured.shape
    grid_size = int(np.sqrt(num_patches))

    # Convert original patches to flat for comparison
    original_patches_flat = original_patches_structured.reshape(batch_size, num_patches, -1)

    # Calculate differences
    rope_diff = rope_patches_flat - original_patches_flat
    string_diff = string_patches_flat - original_patches_flat

    # Reshape differences back to patch structure and take mean across channels
    rope_diff_structured = rope_diff.reshape(batch_size, num_patches, channels, patch_size, patch_size)
    string_diff_structured = string_diff.reshape(batch_size, num_patches, channels, patch_size, patch_size)

    # Create difference heatmaps (mean across color channels)
    rope_heatmap = rope_diff_structured.mean(dim=2)[0]  # [196, 16, 16]
    string_heatmap = string_diff_structured.mean(dim=2)[0]  # [196, 16, 16]

    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original image
    axes[0].imshow(display_img)
    axes[0].set_title('Original Image\n224 x 224', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    def create_difference_grid(diff_patches, ax, title, cmap='RdBu'):
        # Create canvas for difference heatmap
        diff_canvas = torch.zeros(grid_size * patch_size, grid_size * patch_size)

        for i in range(grid_size):
            for j in range(grid_size):
                idx = i * grid_size + j
                patch_diff = diff_patches[idx]  # [16, 16]

                y_start, y_end = i * patch_size, (i + 1) * patch_size
                x_start, x_end = j * patch_size, (j + 1) * patch_size
                diff_canvas[y_start:y_end, x_start:x_end] = patch_diff

        # Display with symmetric color scale
        vmax = max(abs(diff_canvas.min()), abs(diff_canvas.max()))
        im = ax.imshow(diff_canvas.cpu().numpy(), cmap=cmap, vmin=-vmax, vmax=vmax)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')

        # Add colorbar
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        # Add grid lines
        for i in range(1, grid_size):
            ax.axhline(y=i * patch_size - 0.5, color='black', linewidth=0.5, alpha=0.3)
            ax.axvline(x=i * patch_size - 0.5, color='black', linewidth=0.5, alpha=0.3)

    # RoPE difference heatmap
    create_difference_grid(rope_heatmap, axes[1],
                          'RoPE Encoding Effect\n(Difference from Original)')

    # STRING difference heatmap
    create_difference_grid(string_heatmap, axes[2],
                          'STRING Encoding Effect\n(Difference from Original)')

    plt.tight_layout()
    plt.suptitle('Encoding Effects: Visualizing Changes from Original Patches',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.show()

    return fig

def demo_comparison():
    """Enhanced demo with improved visualizations"""

    # Setup
    batch_size, num_patches = 1, 196  # ViT-Base patch count
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    IMAGE_FILENAME = "dogs.jpg"
    img_tensor, display_img = load_and_preprocess_image(IMAGE_FILENAME)

    if img_tensor is None:
        return

    # Extract patches
    patches_flat, positions, patches_structured = extract_patches(img_tensor)
    patches_flat = patches_flat.to(device)
    positions = positions.to(device)
    dim = patches_flat.shape[-1]

    print(f"Input shape: {patches_flat.shape}")
    print(f"Positions shape: {positions.shape}")
    print(f"Patch dimension: {dim}")

    # Initialize encoders
    rope = RoPE2D(dim).to(device)
    cayley_string = CayleySTRING(dim).to(device)

    # Apply encodings
    with torch.no_grad():
        rope_output = rope(patches_flat, positions)
        cayley_output = cayley_string(patches_flat, positions)

    # Main visualization: Side-by-side comparison
    print("\n🖼️ Visualizing input image and patch grids...")
    visualize_input_and_patch_grids(display_img, patches_structured, rope_output, cayley_output, positions)

    # Difference visualization
    print("\n📊 Visualizing encoding effects...")
    visualize_encoding_differences(display_img, patches_structured, rope_output, cayley_output, positions)

    # Print statistics
    rope_change = torch.norm(rope_output - patches_flat) / torch.norm(patches_flat)
    cayley_change = torch.norm(cayley_output - patches_flat) / torch.norm(patches_flat)

    print(f"\nRelative changes:")
    print(f"RoPE: {rope_change:.4f}")
    print(f"Cayley-STRING: {cayley_change:.4f}")

    # Check orthogonality
    P = cayley_string.get_orthogonal_matrix()
    is_orthogonal = torch.allclose(P @ P.T, torch.eye(dim, device=device), atol=1e-3)
    determinant = torch.det(P)

    print(f"\nCayley matrix properties:")
    print(f"Is orthogonal: {is_orthogonal}")
    print(f"Determinant: {determinant:.4f}")

    # Cosine similarity analysis
    cos_sim_rope = torch.cosine_similarity(patches_flat.flatten(), rope_output.flatten(), dim=0)
    cos_sim_cayley = torch.cosine_similarity(patches_flat.flatten(), cayley_output.flatten(), dim=0)

    print(f"\nCosine similarity with original:")
    print(f"RoPE: {cos_sim_rope:.4f}")
    print(f"Cayley-STRING: {cos_sim_cayley:.4f}")

    return rope, cayley_string, patches_flat, positions


if __name__ == "__main__":
    print("🚀 Cayley-STRING vs RoPE Demo")
    print("=" * 40)

    # Main comparison
    rope, cayley_string, patches, positions = demo_comparison()

    print(f"\n🎯 Key Takeaways:")
    print(f"• Cayley-STRING extends RoPE with learnable orthogonal basis")
    print(f"• Maintains all benefits of RoPE (translation invariance, separability)")
    print(f"• Adds adaptability with minimal parameter overhead")
    print(f"• Orthogonality is preserved during training via Cayley parameterization")
    print(f"• Should improve performance on vision tasks requiring spatial reasoning")

# [2D-Positional-Encoding-Vision-Transformer](https://github.com/s-chh/2D-Positional-Encoding-Vision-Transformer) for testing diferent Positional Encoder

In [ ]:
!git clone https://github.com/1kaiser/2D-Positional-Encoding-Vision-Transformer
%cd 2D-Positional-Encoding-Vision-Transformer

# 🏗️ Positional Encoding Architecture Comparison

## 1. 🚫 **NO POSITION** (Baseline)
```
Input Patches → Embedding → Q,K,V → Attention → Output
    [x1]           [e1]      ↓       ↓     ↓
    [x2]     →     [e2]   [q1] [k1] [v1]  
    [x3]           [e3]   [q2] [k2] [v2]  → Softmax(QK^T)V
    [x4]           [e4]   [q3] [k3] [v3]
    
📝 No positional information - tokens are position-agnostic
```

## 2. 📚 **LEARNABLE** Position Encoding
```
Input Patches → Embedding + Learned PE → Q,K,V → Attention → Output
    [x1]         [e1] + [p1]             ↓       ↓     ↓
    [x2]    →    [e2] + [p2]          [q1] [k1] [v1]  
    [x3]         [e3] + [p3]          [q2] [k2] [v2]  → Softmax(QK^T)V
    [x4]         [e4] + [p4]          [q3] [k3] [v3]
    
📝 Adds learned position embeddings [p1,p2,p3,p4] to input
```

## 3. 🌊 **SINUSOIDAL** Position Encoding
```
Input Patches → Embedding + Sin/Cos PE → Q,K,V → Attention → Output
    [x1]         [e1] + [sin(1)]         ↓       ↓     ↓
    [x2]    →    [e2] + [sin(2)]      [q1] [k1] [v1]  
    [x3]         [e3] + [sin(3)]      [q2] [k2] [v2]  → Softmax(QK^T)V
    [x4]         [e4] + [sin(4)]      [q3] [k3] [v3]
    
📝 Uses fixed sinusoidal patterns - no learned parameters
```

## 4. 🔗 **RELATIVE** Position Encoding
```
Input Patches → Embedding → Q,K,V → Modified Attention → Output
    [x1]           [e1]      ↓       ↓     ↓
    [x2]     →     [e2]   [q1] [k1] [v1]  
    [x3]           [e3]   [q2] [k2] [v2]  → Softmax(QK^T + R)V
    [x4]           [e4]   [q3] [k3] [v3]     ↑
                                            Relative Bias Matrix R
                                            [r11 r12 r13 r14]
                                            [r21 r22 r23 r24]
                                            [r31 r32 r33 r34]
                                            [r41 r42 r43 r44]

📝 Modifies attention with learned relative position bias
```

## 5. 🔄 **RoPE** (Rotary Position Encoding)
```
Input Patches → Embedding → Q,K,V → Rotated Q,K → Attention → Output
    [x1]           [e1]      ↓   ↓      ↓    ↓
    [x2]     →     [e2]   [q1][k1]   [q1'] [k1']
    [x3]           [e3]   [q2][k2] → [q2'] [k2'] → Softmax(Q'K'^T)V
    [x4]           [e4]   [q3][k3]   [q3'] [k3']
                                       ↑      ↑
                                   ┌─────────┴─────────┐
                                   │ Rotary Transform  │
                                   │ q' = R(pos) * q   │
                                   │ k' = R(pos) * k   │
                                   └───────────────────┘

📝 Applies position-dependent rotations to Q and K vectors
```

## 6. 🚀 **STRING** (Our Implementation)
```
Input → Embedding → Q,K,V → 2D Split → RoPE → Learnable P → Attention → Output
 [x1]     [e1]      ↓   ↓     ↓        ↓         ↓
 [x2] →   [e2]   [q1][k1]   X|Y    R(pos)    P*R(pos)
 [x3]     [e3]   [q2][k2] → q_x  →  q_x'   →   q_x''
 [x4]     [e4]   [q3][k3]   q_y     q_y'      q_y''     → Softmax(Q''K''^T)V
                              ↓        ↓         ↓
                            k_x  →  k_x'   →   k_x''
                            k_y     k_y'      k_y''
                                      ↑         ↑
                            ┌─────────┴─────────┴─────────┐
                            │ STRING Transformation:      │
                            │ 1. Split dims into X,Y      │
                            │ 2. Apply RoPE to each       │
                            │ 3. Apply learnable matrix P │
                            │ 4. Combine results          │
                            └─────────────────────────────┘

📝 Extends RoPE with learnable transformations for better 2D understanding
```

## 📊 **Detailed STRING Architecture**

```
┌──────────────────────────────────────────────────────────────────┐
│                          STRING ARCHITECTURE                     │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  Input Token [B, S, E]                                          │
│        │                                                        │
│        ▼                                                        │
│  ┌─────────────┐                                                │
│  │   Q, K, V   │  Linear projections                            │
│  │ [B,H,S,HE]  │                                                │
│  └─────────────┘                                                │
│        │                                                        │
│        ▼                                                        │
│  ┌─────────────┐  Split head embedding dimension                │
│  │ Q_x  │  Q_y │  HE = HE/2 + HE/2                             │
│  │[B,H,S│,HE/2]│                                                │
│  │ K_x  │  K_y │                                                │
│  └─────────────┘                                                │
│        │                                                        │
│        ▼                                                        │
│  ┌─────────────┐  For X-axis:                                   │
│  │STRING_X(Q_x)│  1. Get X positions [0,1,1,2,2,3,3,...]       │
│  │STRING_X(K_x)│  2. Apply RoPE rotation                        │
│  └─────────────┘  3. Apply learnable matrix P_x                 │
│        │                                                        │
│        ▼                                                        │
│  ┌─────────────┐  For Y-axis:                                   │
│  │STRING_Y(Q_y)│  1. Get Y positions [0,1,2,3,1,2,3,...]       │
│  │STRING_Y(K_y)│  2. Apply RoPE rotation                        │
│  └─────────────┘  3. Apply learnable matrix P_y                 │
│        │                                                        │
│        ▼                                                        │
│  ┌─────────────┐  Concatenate results                           │
│  │   Q'', K''  │  Q'' = [Q_x'', Q_y'']                         │
│  │ [B,H,S,HE]  │  K'' = [K_x'', K_y'']                         │
│  └─────────────┘                                                │
│        │                                                        │
│        ▼                                                        │
│  ┌─────────────┐  Standard attention                            │
│  │ Attention   │  Softmax(Q''K''^T / √d) V                      │
│  │   Output    │                                                │
│  └─────────────┘                                                │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

## 🔍 **Key Architectural Differences**

| Method | Where Applied | Learnable? | 2D Aware? | Complexity |
|--------|---------------|------------|-----------|------------|
| **No Position** | None | ❌ | ❌ | O(1) |
| **Learnable** | Input embedding | ✅ | ❌ | O(S*E) |
| **Sinusoidal** | Input embedding | ❌ | ❌ | O(1) |
| **Relative** | Attention matrix | ✅ | ✅ | O(S²*E) |
| **RoPE** | Q,K vectors | ❌ | ✅ | O(S*E) |
| **STRING** | Q,K vectors + Transform | ✅ | ✅ | O(S*E) |

## 🎯 **Why STRING is Different**

```
Traditional Methods:          RoPE:                    STRING:
     ┌─────┐                      ┌─────┐                  ┌─────┐
     │  +  │ Add PE               │  ×  │ Rotate           │  ×  │ Rotate
     │ PE  │ to input             │RoPE │ Q,K              │RoPE │ Q,K
     └─────┘                      └─────┘                  └─────┘
        │                           │                         │
        ▼                           ▼                         ▼
   ┌─────────┐                 ┌─────────┐                ┌─────┐
   │Standard │                 │Standard │                │  ×  │ Learn
   │Attention│                 │Attention│                │  P  │ Transform
   └─────────┘                 └─────────┘                └─────┘
                                                             │
                                                             ▼
                                                        ┌─────────┐
                                                        │Standard │
                                                        │Attention│
                                                        └─────────┘
```

## 💡 **STRING's Innovation**

1. **Extends RoPE**: Uses RoPE as foundation, then adds learnable improvements
2. **2D Spatial Awareness**: Splits embeddings into X/Y components  
3. **Learnable Adaptation**: Matrix P learns dataset-specific transformations
4. **Translation Invariant**: Maintains RoPE's translation properties
5. **Efficient**: Only adds small learnable matrix, not O(S²) complexity

This is why STRING achieves **better performance** - it combines the best of RoPE's geometric properties with learnable adaptation! 🚀

# scripts.sh

In [ ]:
!bash scripts.sh

Ep: 1/5	It: 51/391	batch_loss: 2.2887	batch_accuracy: 14.84%	lr: 0.000065
Ep: 1/5	It: 101/391	batch_loss: 2.1448	batch_accuracy: 19.53%	lr: 0.000129
Ep: 1/5	It: 151/391	batch_loss: 1.9390	batch_accuracy: 27.34%	lr: 0.000193
Ep: 1/5	It: 201/391	batch_loss: 1.8573	batch_accuracy: 29.69%	lr: 0.000257
Ep: 1/5	It: 251/391	batch_loss: 1.6380	batch_accuracy: 42.97%	lr: 0.000321
Ep: 1/5	It: 301/391	batch_loss: 1.7544	batch_accuracy: 36.72%	lr: 0.000385
Ep: 1/5	It: 351/391	batch_loss: 1.5989	batch_accuracy: 39.06%	lr: 0.000449
Ep: 1/5	It: 391/391	batch_loss: 1.7469	batch_accuracy: 35.00%	lr: 0.000500
Epoch 1 finished in 41.93 seconds.
Test acc: 40.93%	Test loss: 1.6255
Best test acc: 40.93%

Model parameters saved to ./model/cifar10/ViT_model_sinusoidal.params
Ep: 2/5	It: 1/391	batch_loss: 1.8788	batch_accuracy: 38.28%	lr: 0.000500
Ep: 2/5	It: 51/391	batch_loss: 1.7031	batch_accuracy: 38.28%	lr: 0.000498
Ep: 2/5	It: 101/391	batch_loss: 1.6730	batch_accuracy: 35.94%	lr: 0.000491
Ep: 2/5	It: 151/

```ascii
=== CIFAR10 Test Accuracy Comparison ===

NONE            |████████████████████████▌                   | 49.23%
LEARN           |███████████████████████████                 | 54.14%
SINUSOIDAL      |███████████████████████████▍                | 54.76%
RELATIVE        |█████████████████████████▎                  | 50.53%
ROPE            |█████████████████████████████               | 57.93% ⭐
STRING_CAYLEY   |████████████████████████████▋               | 57.30%
STRING_CIRCULANT|███████████████████████████▉                | 55.73%

    0%    10%    20%    30%    40%    50%    60%    70%    80%


=== CIFAR100 Test Accuracy Comparison ===

NONE            |██████████▌                                 | 21.11%
LEARN           |████████████▏                               | 24.32%
SINUSOIDAL      |████████████▌                               | 25.05%
RELATIVE        |██████████▋                                 | 21.39%
ROPE            |████████████▊                               | 25.67%
STRING_CAYLEY   |█████████████                               | 25.96% ⭐
STRING_CIRCULANT|████████████▏                               | 24.30%

    0%    10%    20%    30%    40%    50%    60%    70%    80%


=== Summary Statistics ===

Method            CIFAR10    CIFAR100    Average
-------------------------------------------------
NONE              49.23%     21.11%      35.17%
LEARN             54.14%     24.32%      39.23%
SINUSOIDAL        54.76%     25.05%      39.91%
RELATIVE          50.53%     21.39%      35.96%
ROPE              57.93%     25.67%      41.80%
STRING_CAYLEY     57.30%     25.96%      41.63% ⭐ Best Overall
STRING_CIRCULANT  55.73%     24.30%      40.02%

📊 Key Findings:
- RoPE performs best on CIFAR10 (57.93%)
- STRING-Cayley performs best on CIFAR100 (25.96%)
- STRING-Cayley has the best average performance (41.63%)
- All position encoding methods outperform no encoding (NONE)
- STRING variants show competitive performance with established methods
```

```ascii
CIFAR10 Results (5 epochs):
NONE: 49.2% | LEARN: 54.1% | SINUSOIDAL: 54.8% | RELATIVE: 50.5%
ROPE: 57.9% ⭐ | STRING-CAYLEY: 57.3% | STRING-CIRCULANT: 55.7%

CIFAR100 Results (5 epochs):
NONE: 21.1% | LEARN: 24.3% | SINUSOIDAL: 25.1% | RELATIVE: 21.4%
ROPE: 25.7% | STRING-CAYLEY: 26.0% ⭐ | STRING-CIRCULANT: 24.3%
```